importing Required modules

In [1]:
import pandas as pd
import numpy as np
import sklearn
from sklearn import preprocessing as per
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt 
from sklearn_pandas import DataFrameMapper
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout
from tensorflow.keras.optimizers import SGD
from sklearn.cross_decomposition import PLSRegression
from lifelines.utils import concordance_index
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error,median_absolute_error


import deepsurvk
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNetCV
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error

G:\ana\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
G:\ana\lib\site-packages\tensorflow_addons\utils\ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.12.0 and strictly below 2.15.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.8.0 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If you want to make sure you're using a tested and supported con

In [2]:
#READING CSV FILE
df1 = pd.read_csv("data_bcr_clinical_data_patient.csv",na_values='?')
#EXCEPT CLINICAL DATA OTHERS HAVE PATIENT IDs WITH -01, SO ADD -01 AT THE END
df1.at[4,"Patient Identifier"]
def ankfunc(s):
    return s+"-01"
for i in range(4,532):
    df1.at[i,"Patient Identifier"]=ankfunc(df1.at[i,"Patient Identifier"])

#DROP ROWS AND COLUMNS
df1.drop([0,1,2,3] , inplace=True)
df1.set_index("Patient Identifier", inplace=True)

    
df1.replace('unknown',np.nan , inplace=True)
df1.replace('[Not Available]',np.nan , inplace=True)

df1.fillna(df1.mean(), inplace=True)
df1

C:\Users\PRODEE~1\AppData\Local\Temp/ipykernel_8288/2649448865.py:18: FutureWarning: Dropping of nuisance columns in DataFrame reductions (with 'numeric_only=None') is deprecated; in a future version this will raise TypeError.  Select only valid columns before calling the reduction.
  df1.fillna(df1.mean(), inplace=True)


,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,Diagnosis Age,Neoplasm American Joint Committee on Cancer Clinical Distant Metastasis M Stage,Neoplasm American Joint Committee on Cancer Clinical Regional Lymph Node N Stage,Neoplasm American Joint Committee on Cancer Clinical Primary Tumor T Stage,Neoplasm American Joint Committee on Cancer Clinical Group Stage,Last Alive Less Initial Pathologic Diagnosis Date Calculated Day Value,Overall Survival Status,Overall Survival (Months),Disease Free Status,Disease Free (Months)
Patient Identifier,,,,,,,,,,,,,,,,,,,,,
TCGA-4P-AA8J-01,Oral Tongue,Male,BLACK OR AFRICAN AMERICAN,NOT HISPANIC OR LATINO,No,No,2013,YES,Alive,7th,...,66,M0,N2a,T4a,Stage IVA,0,0:LIVING,3.35,0:DiseaseFree,3.35
TCGA-BA-4074-01,Oral Tongue,Male,WHITE,NOT HISPANIC OR LATINO,No,No,2003,YES,Dead,6th,...,69,M0,N2c,T3,Stage IVA,0,1:DECEASED,15.18,1:Recurred/Progressed,13.01
TCGA-BA-4075-01,Oral Tongue,Male,BLACK OR AFRICAN AMERICAN,NOT HISPANIC OR LATINO,Yes,Yes,2004,YES,Dead,6th,...,49,M0,N1,T4a,Stage IVA,0,1:DECEASED,9.3,1:Recurred/Progressed,7.75
TCGA-BA-4076-01,Larynx,Male,WHITE,NOT HISPANIC OR LATINO,No,No,2003,YES,Dead,6th,...,39,M0,N2c,T3,Stage IVA,0,1:DECEASED,13.63,1:Recurred/Progressed,9.4
TCGA-BA-4077-01,Base of tongue,Female,WHITE,NOT HISPANIC OR LATINO,Yes,Yes,2003,YES,Dead,6th,...,45,M0,N3,T4b,Stage IVB,0,1:DECEASED,37.25,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-UF-A7JT-01,Floor of mouth,Female,WHITE,NaN,No,No,2009,YES,Dead,6th,...,72,M0,N0,T4a,Stage IVA,0,1:DECEASED,32.62,1:Recurred/Progressed,23.59
TCGA-UF-A7JV-01,Hypopharynx,Female,WHITE,NaN,"Yes, History of Synchronous/Bilateral Malignancy",No,2011,YES,Dead,7th,...,62,M0,N2c,T4a,Stage IVA,0,1:DECEASED,2.96,1:Recurred/Progressed,1.81
TCGA-UP-A6WW-01,Oral Tongue,Male,WHITE,HISPANIC OR LATINO,No,No,2013,YES,Alive,7th,...,58,MX,N2c,T2,Stage IVA,0,0:LIVING,17.02,0:DiseaseFree,17.02


In [3]:
df1.fillna(method='ffill', inplace=True)
df1.fillna(method='bfill', inplace=True)
df1

,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,Diagnosis Age,Neoplasm American Joint Committee on Cancer Clinical Distant Metastasis M Stage,Neoplasm American Joint Committee on Cancer Clinical Regional Lymph Node N Stage,Neoplasm American Joint Committee on Cancer Clinical Primary Tumor T Stage,Neoplasm American Joint Committee on Cancer Clinical Group Stage,Last Alive Less Initial Pathologic Diagnosis Date Calculated Day Value,Overall Survival Status,Overall Survival (Months),Disease Free Status,Disease Free (Months)
Patient Identifier,,,,,,,,,,,,,,,,,,,,,
TCGA-4P-AA8J-01,Oral Tongue,Male,BLACK OR AFRICAN AMERICAN,NOT HISPANIC OR LATINO,No,No,2013,YES,Alive,7th,...,66,M0,N2a,T4a,Stage IVA,0,0:LIVING,3.35,0:DiseaseFree,3.35
TCGA-BA-4074-01,Oral Tongue,Male,WHITE,NOT HISPANIC OR LATINO,No,No,2003,YES,Dead,6th,...,69,M0,N2c,T3,Stage IVA,0,1:DECEASED,15.18,1:Recurred/Progressed,13.01
TCGA-BA-4075-01,Oral Tongue,Male,BLACK OR AFRICAN AMERICAN,NOT HISPANIC OR LATINO,Yes,Yes,2004,YES,Dead,6th,...,49,M0,N1,T4a,Stage IVA,0,1:DECEASED,9.3,1:Recurred/Progressed,7.75
TCGA-BA-4076-01,Larynx,Male,WHITE,NOT HISPANIC OR LATINO,No,No,2003,YES,Dead,6th,...,39,M0,N2c,T3,Stage IVA,0,1:DECEASED,13.63,1:Recurred/Progressed,9.4
TCGA-BA-4077-01,Base of tongue,Female,WHITE,NOT HISPANIC OR LATINO,Yes,Yes,2003,YES,Dead,6th,...,45,M0,N3,T4b,Stage IVB,0,1:DECEASED,37.25,1:Recurred/Progressed,9.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-UF-A7JT-01,Floor of mouth,Female,WHITE,NOT HISPANIC OR LATINO,No,No,2009,YES,Dead,6th,...,72,M0,N0,T4a,Stage IVA,0,1:DECEASED,32.62,1:Recurred/Progressed,23.59
TCGA-UF-A7JV-01,Hypopharynx,Female,WHITE,NOT HISPANIC OR LATINO,"Yes, History of Synchronous/Bilateral Malignancy",No,2011,YES,Dead,7th,...,62,M0,N2c,T4a,Stage IVA,0,1:DECEASED,2.96,1:Recurred/Progressed,1.81
TCGA-UP-A6WW-01,Oral Tongue,Male,WHITE,HISPANIC OR LATINO,No,No,2013,YES,Alive,7th,...,58,MX,N2c,T2,Stage IVA,0,0:LIVING,17.02,0:DiseaseFree,17.02


In [4]:
df1['Lymph node neck dissection indicator'].replace(['[Not Available]','NO','YES'],['00','1','2'],inplace=True)

df1['Overall Survival Status'].replace(['0:LIVING','1:DECEASED'],['0','1'],inplace=True)
df1['Patient Primary Tumor Site'].replace(['[Not Available]','Buccal Mucosa','Larynx','Oral Cavity','Floor of mouth','Tonsil','Hypopharynx','Alveolar Ridge','Hard Palate','Oropharynx','Lip','Base of tongue','Oral Tongue'],['00','1','2','3','4','5','6','7','8','9','10','11','12'],inplace=True)
df1['Sex'].replace(['[Not Available]','Male','Female'],['00','1','2'],inplace=True)
df1['Race Category'].replace(['[Not Available]','WHITE','BLACK OR AFRICAN AMERICAN','ASIAN','AMERICAN INDIAN OR ALASKA NATIVE'],['00','1','2','3','4'],inplace=True)
df1['Ethnicity Category'].replace(['[Not Available]','NOT HISPANIC OR LATINO','HISPANIC OR LATINO'],['00','1','2'],inplace=True)
df1['Prior Cancer Diagnosis Occurence'].replace(['[Not Available]','No','Yes','Yes, History of Synchronous/Bilateral Malignancy','Yes, History of Prior Malignancy'],['00','1','2','3','4'],inplace=True)
df1['Neoadjuvant Therapy Type Administered Prior To Resection Text'].replace(['[Not Available]','No','Yes'],['00','1','2'],inplace=True)
df1['Vital Status'].replace(['[Not Available]','Dead','Alive'],['00','1','2'],inplace=True)
df1['American Joint Committee on Cancer Publication Version Type'].replace(['[Not Available]','6th','7th','5th','4th'],['00','1','2','3','4'],inplace=True)
df1['American Joint Committee on Cancer Tumor Stage Code'].replace(['[Not Available]','T0','T1','T2','T3','T4','T4a','T4b','TX'],['00','1','2','3','4','5','6','7','8'],inplace=True)
df1['Disease Free Status'].replace(['0:DiseaseFree','1:Recurred/Progressed','[Not Available]'],['0','1','00'],inplace=True)
df1['Neoplasm Histologic Grade'].replace(['[Not Available]','G1','G2','G3','GX','G4'],['00','1','2','3','4','5'],inplace=True)
df1['Alcohol History Documented'].replace(['[Not Available]','No','Yes','NO','YES'],['00','1','2','3','4'],inplace=True)
df1['Neoplasm Disease Lymph Node Stage American Joint Committee on Cancer Code'].replace(['[Not Available]','N0','N1','N2','N2a','N2b','N2c','N3','NX'],['00','1','2','3','4','5','6','7','8'],inplace=True)
df1['Neoplasm Disease Stage American Joint Committee on Cancer Code'].replace(['[Not Available]','Discrepancy','Stage I','Stage II','Stage III','Stage IVA','Stage IVB','Stage IVC'],['00','00','1','2','3','4','5','6'],inplace=True)
df1['Neoplasm American Joint Committee on Cancer Clinical Distant Metastasis M Stage'].replace(['[Not Available]','M1','M1','MX','M0'],['00','1','2','3','4'],inplace=True)
df1['Neoplasm American Joint Committee on Cancer Clinical Regional Lymph Node N Stage'].replace(['[Not Available]','N0','N1','N2a','N2b','N2c','N3','NX','N2'],['00','1','2','3','4','5','6','7','8'],inplace=True)
df1['Neoplasm American Joint Committee on Cancer Clinical Primary Tumor T Stage'].replace(['[Not Available]','T1','T2','T3','T4a','T4b','TX','T4'],['00','1','2','3','4','5','6','7'],inplace=True)
df1['Neoplasm American Joint Committee on Cancer Clinical Group Stage'].replace(['[Not Available]','Stage I','Stage II','Stage III','Stage IVA','Stage IVB','Stage IVC'],['00','1','2','3','4','5','6'],inplace=True)

df1.head()

,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,Diagnosis Age,Neoplasm American Joint Committee on Cancer Clinical Distant Metastasis M Stage,Neoplasm American Joint Committee on Cancer Clinical Regional Lymph Node N Stage,Neoplasm American Joint Committee on Cancer Clinical Primary Tumor T Stage,Neoplasm American Joint Committee on Cancer Clinical Group Stage,Last Alive Less Initial Pathologic Diagnosis Date Calculated Day Value,Overall Survival Status,Overall Survival (Months),Disease Free Status,Disease Free (Months)
Patient Identifier,,,,,,,,,,,,,,,,,,,,,
TCGA-4P-AA8J-01,12,1,2,1,1,1,2013,2,2,2,...,66,4,3,4,4,0,0,3.35,0,3.35
TCGA-BA-4074-01,12,1,1,1,1,1,2003,2,1,1,...,69,4,5,3,4,0,1,15.18,1,13.01
TCGA-BA-4075-01,12,1,2,1,2,2,2004,2,1,1,...,49,4,2,4,4,0,1,9.3,1,7.75
TCGA-BA-4076-01,2,1,1,1,1,1,2003,2,1,1,...,39,4,5,3,4,0,1,13.63,1,9.4
TCGA-BA-4077-01,11,2,1,1,2,2,2003,2,1,1,...,45,4,6,5,5,0,1,37.25,1,9.4


In [5]:
#STORING REDUCED DATA TO CSV
cl = pd.DataFrame(df1)
cl.to_csv("CLINICALpreprocessed.csv")
print("Data exported to csv file")

Data exported to csv file


In [6]:
#READING CSV FILE
df =pd.read_csv("data_methylation_hm450.csv")

#DROP ROWS AND COLUMNS
df =df.dropna()
df.drop("Entrez_Gene_Id", axis=1 ,inplace=True)

#SET INDEX
df.set_index('Hugo_Symbol', inplace=True)

#TRANSPOSE ROWS INTO COLUMNS
df2 = df.T
df2.to_csv("Methylation.csv")
df2.head()

Hugo_Symbol,TSEN34,MUSTN1,C3orf16,CKLF,SFRS7,FAM180B,PTPRF,C6orf168,LOC728024,DSTYK,...,VDAC1,DDX46,FAM13B,IL12B,PHACTR2,PRKRIP1,AGK,EZH2,AUH,ZNF189
TCGA-4P-AA8J-01,0.160658,0.856690,0.689981,0.063268,0.088333,0.738956,0.689032,0.485472,0.847822,0.018144,...,0.021239,0.095778,0.061835,0.043785,0.055341,0.066497,0.080675,0.041548,0.054317,0.098915
TCGA-BA-4074-01,0.172720,0.888797,0.448310,0.095680,0.054274,0.506644,0.842746,0.188788,0.916802,0.018413,...,0.036593,0.086075,0.057365,0.059737,0.047445,0.075115,0.154639,0.076333,0.070420,0.097873
TCGA-BA-4075-01,0.091838,0.876359,0.336352,0.079018,0.062922,0.475571,0.783786,0.221566,0.792347,0.021204,...,0.034351,0.100891,0.048480,0.053835,0.059135,0.095026,0.116404,0.073396,0.082182,0.077774
TCGA-BA-4076-01,0.127324,0.911893,0.757925,0.095460,0.073372,0.834641,0.718938,0.791580,0.898727,0.014496,...,0.024872,0.075242,0.044595,0.050950,0.026872,0.111361,0.222005,0.057531,0.038989,0.064009
TCGA-BA-4077-01,0.132946,0.893790,0.556940,0.074819,0.080927,0.773512,0.392731,0.283078,0.857577,0.018026,...,0.034238,0.067696,0.043182,0.051532,0.054207,0.148919,0.097344,0.052320,0.045407,0.069227


In [7]:
# Merge datasets based on the patient identifier
merged_df = pd.merge(cl, df2, left_index=True, right_index=True, how="inner")
df2=merged_df


In [8]:

c2 = pd.DataFrame(df2)
c2.to_csv("Methylationpreprocessed.csv")

In [9]:

# Extract target variable (survival time) from clinical data
y = c2['Overall Survival (Months)']
c2 = c2.drop('Overall Survival (Months)', axis=1)


In [10]:
y.drop(y.index[-1], inplace=True)
dm=c2.iloc[:,:]
#print(d)
dm = dm.reset_index()
M=dm.iloc[1:,1:]
M


,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,VDAC1,DDX46,FAM13B,IL12B,PHACTR2,PRKRIP1,AGK,EZH2,AUH,ZNF189
1,12,1,1,1,1,1,2003,2,1,1,...,0.036593,0.086075,0.057365,0.059737,0.047445,0.075115,0.154639,0.076333,0.070420,0.097873
2,12,1,2,1,2,2,2004,2,1,1,...,0.034351,0.100891,0.048480,0.053835,0.059135,0.095026,0.116404,0.073396,0.082182,0.077774
3,2,1,1,1,1,1,2003,2,1,1,...,0.024872,0.075242,0.044595,0.050950,0.026872,0.111361,0.222005,0.057531,0.038989,0.064009
4,11,2,1,1,2,2,2003,2,1,1,...,0.034238,0.067696,0.043182,0.051532,0.054207,0.148919,0.097344,0.052320,0.045407,0.069227
5,2,1,1,1,1,1,2003,2,1,1,...,0.036186,0.065871,0.042553,0.043962,0.036412,0.131708,0.062926,0.043616,0.044998,0.057528
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
523,4,2,1,1,1,1,2009,2,1,1,...,0.019317,0.045418,0.032703,0.032664,0.035815,0.044621,0.080887,0.037113,0.033082,0.048041
524,6,2,1,1,3,1,2011,2,1,2,...,0.030936,0.068638,0.042534,0.043967,0.047700,0.063594,0.094378,0.045045,0.049718,0.084455
525,12,1,1,2,1,1,2013,2,2,2,...,0.024656,0.044608,0.030440,0.047566,0.064576,0.062233,0.051034,0.034986,0.053662,0.051514
526,4,1,1,1,1,1,2012,2,1,2,...,0.026150,0.078658,0.047960,0.048287,0.050811,0.059785,0.077223,0.051496,0.048270,0.059510


In [11]:
#standardize the data
scaler = StandardScaler()
X1 = scaler.fit_transform(M[:])
#PCA 
# fit pca on data
pca = PLSRegression(n_components=387)
pca.fit(X1,y)
Z1=pca.transform(X1)



G:\ana\lib\site-packages\sklearn\cross_decomposition\_pls.py:348: UserWarning: y residual is constant at iteration 327
  warnings.warn(f"y residual is constant at iteration {k}")


In [12]:
n1 = pd.DataFrame(Z1)
print(n1.shape)
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
n1["Hugo"]=hugo
n1

(527, 387)


,0,1,2,3,4,5,6,7,8,9,...,378,379,380,381,382,383,384,385,386,Hugo
0,0.053190,21.767112,-15.168404,20.437665,6.468261,32.143517,-23.241346,-8.462181,-12.086400,-8.259289,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4074-01
1,-10.089159,11.037341,15.295528,56.358812,-4.790379,10.326288,41.336410,-1.728282,17.755228,-5.264010,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4075-01
2,6.819391,-40.061646,-12.702422,11.427975,-4.843945,6.282527,26.621898,-29.574243,7.007190,-10.518041,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4076-01
3,5.464419,-0.783242,-12.501719,6.013811,-7.452908,-4.760014,12.722819,-12.436072,2.529993,27.580584,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4077-01
4,19.781663,12.003652,-21.319956,-7.880330,-3.633865,1.122350,3.755521,-7.656339,43.088505,17.058925,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,-16.902959,-15.409048,17.617336,-16.601228,-2.904980,-2.056018,11.051597,-1.343254,-28.472771,-4.318834,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-UF-A7JT-01
523,-24.670826,-29.992810,1.034167,10.010121,14.378863,-23.493218,17.672512,13.437419,-10.679313,-8.099651,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-UF-A7JV-01
524,-18.683781,-2.764322,-3.543708,-21.346359,-5.170680,10.710463,-2.157883,-20.110333,8.246791,30.379481,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-UP-A6WW-01
525,-44.627733,43.549429,29.684673,-17.168742,-39.154742,-50.388135,27.279898,-20.658329,-17.126615,-16.093342,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-WA-A7GZ-01


In [13]:

lasso = Lasso(alpha=0.1)  # alpha is the regularization strength
lasso.fit(Z1, y)
n_components = Z1.shape[1]
feature_names = [i+1 for i in range(n_components)]
coef = pd.Series(lasso.coef_, index=feature_names)
selected_features = coef.abs().nlargest(348).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z1[:, i]
        columns.append(column)
except Exception:
    pass
Zl1 = np.column_stack(columns)
nl1 = pd.DataFrame(Zl1)
print(nl1.shape)
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
nl1["Hugo"]=hugo
nl1

Number of selected features:  348
Selected features:  Int64Index([  5,   1,   3,   8,   6,  13,   4,  10,   2,  12,
            ...
            339, 340, 341, 342, 343, 344, 345, 346, 347, 348],
           dtype='int64', length=348)
(527, 348)


,0,1,2,3,4,5,6,7,8,9,...,339,340,341,342,343,344,345,346,347,Hugo
0,32.143517,21.767112,20.437665,-12.086400,-23.241346,7.810301,6.468261,11.133056,-15.168404,5.530465,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4074-01
1,10.326288,11.037341,56.358812,17.755228,41.336410,-0.871812,-4.790379,15.111963,15.295528,5.518289,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4075-01
2,6.282527,-40.061646,11.427975,7.007190,26.621898,-6.538156,-4.843945,-7.054875,-12.702422,9.553150,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4076-01
3,-4.760014,-0.783242,6.013811,2.529993,12.722819,11.859804,-7.452908,5.137253,-12.501719,-1.977588,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4077-01
4,1.122350,12.003652,-7.880330,43.088505,3.755521,-0.547351,-3.633865,9.872085,-21.319956,5.121771,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,-2.056018,-15.409048,-16.601228,-28.472771,11.051597,-13.637890,-2.904980,-15.511747,17.617336,1.543088,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-UF-A7JT-01
523,-23.493218,-29.992810,10.010121,-10.679313,17.672512,2.934618,14.378863,3.390020,1.034167,4.895245,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-UF-A7JV-01
524,10.710463,-2.764322,-21.346359,8.246791,-2.157883,6.597183,-5.170680,-41.242691,-3.543708,2.230767,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-UP-A6WW-01
525,-50.388135,43.549429,-17.168742,-17.126615,27.279898,6.988775,-39.154742,-18.143436,29.684673,-2.030798,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-WA-A7GZ-01


In [14]:
en = ElasticNetCV(l1_ratio=0.5, cv=5)
en.fit(Z1, y)
coef = pd.Series(en.coef_, index=[i+1 for i in range(Z1.shape[1])])
selected_features = coef.abs().nlargest(348).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())
try:
    columns = []
    for i in selected_features:
        column = Z1[:, i]
        columns.append(column)
except Exception:
    pass
Ze1 = np.column_stack(columns)
ne1 = pd.DataFrame(Ze1)
ne1.shape
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
ne1["Hugo"]=hugo
ne1


Number of selected features:  348
Selected features:  [5, 1, 3, 8, 6, 13, 4, 10, 2, 12, 7, 15, 16, 11, 14, 9, 17, 18, 20, 19, 21, 23, 22, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211

,0,1,2,3,4,5,6,7,8,9,...,339,340,341,342,343,344,345,346,347,Hugo
0,32.143517,21.767112,20.437665,-12.086400,-23.241346,7.810301,6.468261,11.133056,-15.168404,5.530465,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4074-01
1,10.326288,11.037341,56.358812,17.755228,41.336410,-0.871812,-4.790379,15.111963,15.295528,5.518289,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4075-01
2,6.282527,-40.061646,11.427975,7.007190,26.621898,-6.538156,-4.843945,-7.054875,-12.702422,9.553150,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4076-01
3,-4.760014,-0.783242,6.013811,2.529993,12.722819,11.859804,-7.452908,5.137253,-12.501719,-1.977588,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4077-01
4,1.122350,12.003652,-7.880330,43.088505,3.755521,-0.547351,-3.633865,9.872085,-21.319956,5.121771,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,-2.056018,-15.409048,-16.601228,-28.472771,11.051597,-13.637890,-2.904980,-15.511747,17.617336,1.543088,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-UF-A7JT-01
523,-23.493218,-29.992810,10.010121,-10.679313,17.672512,2.934618,14.378863,3.390020,1.034167,4.895245,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-UF-A7JV-01
524,10.710463,-2.764322,-21.346359,8.246791,-2.157883,6.597183,-5.170680,-41.242691,-3.543708,2.230767,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-UP-A6WW-01
525,-50.388135,43.549429,-17.168742,-17.126615,27.279898,6.988775,-39.154742,-18.143436,29.684673,-2.030798,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-WA-A7GZ-01


In [15]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(Z1, y)
importances = rf.feature_importances_
feature_names = [i+1 for i in range(Z1.shape[1])]
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
importance_df.sort_values("Importance", ascending=False, inplace=True)
selected_features = importance_df["Feature"].head(348)
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())
try:
    columns = []
    for i in selected_features:
        column = Z1[:, i]
        columns.append(column)
except Exception:
    pass
Zrf1 = np.column_stack(columns)
nrf1 = pd.DataFrame(Zrf1)
nrf1.shape
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
nrf1["Hugo"]=hugo
nrf1

Number of selected features:  348
Selected features:  [1, 4, 3, 8, 5, 6, 2, 10, 16, 19, 14, 13, 25, 35, 51, 11, 7, 272, 270, 42, 12, 53, 22, 227, 75, 278, 167, 30, 215, 179, 105, 9, 178, 266, 307, 27, 128, 70, 189, 314, 122, 166, 131, 43, 87, 15, 56, 211, 260, 228, 78, 287, 303, 252, 182, 203, 262, 142, 40, 26, 130, 196, 90, 32, 155, 271, 274, 36, 286, 68, 245, 297, 321, 86, 37, 24, 161, 168, 60, 65, 98, 28, 309, 201, 290, 291, 288, 301, 261, 259, 322, 41, 114, 275, 324, 88, 292, 243, 295, 21, 197, 117, 116, 229, 129, 285, 254, 156, 102, 299, 209, 313, 148, 233, 276, 247, 289, 249, 118, 273, 31, 172, 181, 244, 145, 198, 264, 308, 234, 77, 217, 104, 64, 66, 147, 268, 296, 237, 47, 84, 316, 52, 320, 256, 140, 143, 165, 326, 49, 23, 224, 231, 220, 57, 298, 18, 205, 158, 240, 222, 124, 319, 304, 132, 29, 44, 325, 327, 219, 300, 160, 38, 248, 151, 33, 46, 255, 115, 111, 34, 177, 159, 246, 157, 17, 81, 71, 207, 263, 225, 72, 230, 39, 190, 79, 242, 106, 123, 280, 294, 174, 282, 176, 281, 96, 

,0,1,2,3,4,5,6,7,8,9,...,339,340,341,342,343,344,345,346,347,Hugo
0,21.767112,6.468261,20.437665,-12.086400,32.143517,-23.241346,-15.168404,11.133056,5.321723,1.276130,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4074-01
1,11.037341,-4.790379,56.358812,17.755228,10.326288,41.336410,15.295528,15.111963,-1.875346,-28.797978,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4075-01
2,-40.061646,-4.843945,11.427975,7.007190,6.282527,26.621898,-12.702422,-7.054875,-8.803062,2.705197,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4076-01
3,-0.783242,-7.452908,6.013811,2.529993,-4.760014,12.722819,-12.501719,5.137253,0.457845,-0.510615,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4077-01
4,12.003652,-3.633865,-7.880330,43.088505,1.122350,3.755521,-21.319956,9.872085,-5.729991,-15.405945,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,-15.409048,-2.904980,-16.601228,-28.472771,-2.056018,11.051597,17.617336,-15.511747,2.221242,-9.682885,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-UF-A7JT-01
523,-29.992810,14.378863,10.010121,-10.679313,-23.493218,17.672512,1.034167,3.390020,1.678876,2.514907,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-UF-A7JV-01
524,-2.764322,-5.170680,-21.346359,8.246791,10.710463,-2.157883,-3.543708,-41.242691,2.131879,13.961988,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-UP-A6WW-01
525,43.549429,-39.154742,-17.168742,-17.126615,-50.388135,27.279898,29.684673,-18.143436,-28.816922,6.962603,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-WA-A7GZ-01


In [16]:
lr = LinearRegression()
rfe = RFE(lr, n_features_to_select=348)
rfe.fit(Z1, y)
selected_features = [i+1 for i in range(len(rfe.support_)) if rfe.support_[i]]
print("Selected Features:", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z1[:, i]
        columns.append(column)
except Exception:
    pass
Zre1 = np.column_stack(columns)
nre1 = pd.DataFrame(Zre1)
nre1.shape
hugo=[]
for i in range(1,528):
    hugo.append(dm['index'][i])
nre1["Hugo"]=hugo
nre1

Selected Features: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218

,0,1,2,3,4,5,6,7,8,9,...,339,340,341,342,343,344,345,346,347,Hugo
0,21.767112,-15.168404,20.437665,6.468261,32.143517,-23.241346,-8.462181,-12.086400,-8.259289,11.133056,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4074-01
1,11.037341,15.295528,56.358812,-4.790379,10.326288,41.336410,-1.728282,17.755228,-5.264010,15.111963,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4075-01
2,-40.061646,-12.702422,11.427975,-4.843945,6.282527,26.621898,-29.574243,7.007190,-10.518041,-7.054875,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4076-01
3,-0.783242,-12.501719,6.013811,-7.452908,-4.760014,12.722819,-12.436072,2.529993,27.580584,5.137253,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4077-01
4,12.003652,-21.319956,-7.880330,-3.633865,1.122350,3.755521,-7.656339,43.088505,17.058925,9.872085,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522,-15.409048,17.617336,-16.601228,-2.904980,-2.056018,11.051597,-1.343254,-28.472771,-4.318834,-15.511747,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-UF-A7JT-01
523,-29.992810,1.034167,10.010121,14.378863,-23.493218,17.672512,13.437419,-10.679313,-8.099651,3.390020,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-UF-A7JV-01
524,-2.764322,-3.543708,-21.346359,-5.170680,10.710463,-2.157883,-20.110333,8.246791,30.379481,-41.242691,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-UP-A6WW-01
525,43.549429,29.684673,-17.168742,-39.154742,-50.388135,27.279898,-20.658329,-17.126615,-16.093342,-18.143436,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TCGA-WA-A7GZ-01


In [17]:
#READING CSV FILE
df = pd.read_csv("data_RNA_Seq_v2_expression_median.csv")

#DROP ROWS AND COLUMNS
df=df.replace(0,np.nan)
df=df.dropna()
df=df.replace(np.nan,0)
df.drop("Entrez_Gene_Id", axis=1 ,inplace=True)

#SET INDEX
df.set_index('Hugo_Symbol', inplace=True)

#TRANSPOSE ROWS INTO COLUMNS
df4 = df.T
df4.to_csv("RNAb.csv")
df4.head()
merged_df = pd.merge(cl, df4, left_index=True, right_index=True, how="inner")
df4=merged_df

In [18]:
c4 = pd.DataFrame(df4)
c4.to_csv("RNApreprocessed.csv")
y = c4['Overall Survival (Months)']
c4 = c4.drop('Overall Survival (Months)', axis=1)
y.drop(y.index[-1], inplace=True)
drn=c4.iloc[:,:]
#print(d)
drn = drn.reset_index()
Rn=drn.iloc[1:,1:]
Rn.head()


,Patient Primary Tumor Site,Sex,Race Category,Ethnicity Category,Prior Cancer Diagnosis Occurence,Neoadjuvant Therapy Type Administered Prior To Resection Text,Year Cancer Initial Diagnosis,Lymph node neck dissection indicator,Vital Status,American Joint Committee on Cancer Publication Version Type,...,LOC154274,ZW10,ZWILCH,ZWINT,ZXDB,LOC100130182,ZYG11B,ZYX,FLJ10821,ZZZ3
1,12,1,1,1,1,1,2003,2,1,1,...,311.0030,283.6409,2132.4595,1193.1417,172.2656,380.3717,805.0624,2516.9279,258.5911,1088.3179
2,12,1,2,1,2,2,2004,2,1,1,...,225.1105,512.3945,761.0023,673.1877,172.0488,562.2404,487.7395,5930.0549,292.6437,980.3028
3,2,1,1,1,1,1,2003,2,1,1,...,157.9431,307.4905,480.0682,1032.6643,324.2818,1440.9025,722.5502,2674.5376,672.1763,998.5570
4,11,2,1,1,2,2,2003,2,1,1,...,137.6323,361.4052,1325.3128,1620.3080,210.7796,1423.0029,770.9336,8035.6112,763.2339,692.9740
5,2,1,1,1,1,1,2003,2,1,1,...,241.8520,414.1231,874.1257,1145.1112,372.9953,2634.2473,780.1345,3895.2406,1556.6477,1309.6223


Applying PCA dimensioality reduction technique

In [19]:
#PCA 
scaler = StandardScaler()
# Fit on training set only.
X3 = scaler.fit_transform(Rn[:])
#fit pca on data
pca = PLSRegression(n_components=323)
pca.fit(X3,y)
#transform pca
Z3 =pca.transform(X3)
Z3


array([[-9.55212585e+01, -6.63545659e+01,  3.76703381e+01, ...,
        -3.03523108e-01, -6.07368350e-01,  4.07874584e-03],
       [-5.89266929e+01, -3.44787524e+01,  1.87475724e+01, ...,
         4.80922222e-01,  5.77560365e-01,  2.74843693e-01],
       [-1.00599405e+01, -1.60784102e+01, -8.61716225e+00, ...,
         2.88731962e-01, -2.24208534e-01,  6.39235973e-03],
       ...,
       [-6.16364669e+01,  7.20225142e+00,  7.42975874e+00, ...,
        -9.39824209e-02, -2.72106007e-01, -4.49099898e-01],
       [-7.72490232e+00, -4.28762943e+00, -6.08543074e+00, ...,
         7.20379449e-01, -2.63863713e-01,  1.17672020e+00],
       [ 2.15246953e+00, -1.43602843e+01, -1.80458118e+01, ...,
        -2.39087190e-01, -1.21836839e+00,  1.83077499e+00]])

Applying RFE Feature selection methods

In [20]:
n3 = pd.DataFrame(Z3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
n3["Hugo"]=hugo
n3

,0,1,2,3,4,5,6,7,8,9,...,314,315,316,317,318,319,320,321,322,Hugo
0,-95.521259,-66.354566,37.670338,67.978323,28.779115,42.128789,5.142525,35.997939,40.859722,53.084359,...,0.403235,0.606136,0.721832,-0.017289,-0.501725,-0.094052,-0.303523,-0.607368,0.004079,TCGA-BA-4074-01
1,-58.926693,-34.478752,18.747572,52.967754,5.816400,31.905618,-0.986095,20.203627,29.976171,36.521362,...,0.730708,0.662740,0.770012,0.955220,0.539900,0.470969,0.480922,0.577560,0.274844,TCGA-BA-4075-01
2,-10.059941,-16.078410,-8.617162,-3.561595,-27.873829,8.703295,-9.908476,12.940778,2.978251,21.559138,...,0.393529,0.010860,-0.098502,0.129597,-0.686471,0.719325,0.288732,-0.224209,0.006392,TCGA-BA-4076-01
3,7.506617,6.949812,10.794430,3.255420,-7.035513,17.515897,-8.598195,6.982973,17.199286,-33.773920,...,-1.195129,-1.515942,0.660014,0.651431,-0.044138,0.188518,0.381581,0.922257,0.367034,TCGA-BA-4077-01
4,16.753130,4.653888,30.849528,-31.948145,17.882788,-0.572337,-9.522095,-9.670178,19.792068,0.859542,...,-0.410290,-0.577895,-0.607617,-0.245760,0.648308,-0.287860,1.031961,-1.287826,-1.785377,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,-44.019656,-5.278254,-8.064500,18.984889,4.774887,17.305591,-4.263380,-0.753947,-5.785138,-6.504918,...,-1.023619,1.692489,-1.189679,-0.174098,-0.761829,0.207168,0.470725,-0.394500,0.428045,TCGA-UF-A7JT-01
515,-12.122815,-6.601919,-19.149224,9.584200,5.542687,13.511837,-20.251779,1.899250,1.880846,6.577262,...,-0.765787,-0.335761,1.210208,-0.526118,-1.393847,0.765847,0.000097,0.943023,1.060078,TCGA-UF-A7JV-01
516,-61.636467,7.202251,7.429759,-34.309695,-4.314327,22.823079,-14.520433,8.153220,-9.384123,-21.273752,...,0.846924,-0.777825,-1.043758,-0.705052,0.262193,-0.423263,-0.093982,-0.272106,-0.449100,TCGA-UP-A6WW-01
517,-7.724902,-4.287629,-6.085431,-20.180291,-17.785339,-2.094106,-0.179792,-14.722472,-7.230530,20.345662,...,-0.605799,-0.951320,0.212750,1.258194,1.386434,0.208172,0.720379,-0.263864,1.176720,TCGA-WA-A7GZ-01


In [21]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(Z3, y)
importances = rf.feature_importances_
feature_names = [i+1 for i in range(Z3.shape[1])]
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
importance_df.sort_values("Importance", ascending=False, inplace=True)
selected_features = importance_df["Feature"].head(291)
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())

try:
    columns = []
    for i in selected_features:
        column = Z3[:, i]
        columns.append(column)
except Exception:
    pass
Zrf3 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
nrf3 = pd.DataFrame(Zrf3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
nrf3["Hugo"]=hugo
nrf3

Number of selected features:  291
Selected features:  [2, 9, 3, 6, 4, 7, 1, 5, 8, 14, 311, 11, 19, 259, 315, 13, 159, 10, 322, 15, 300, 161, 237, 260, 16, 232, 170, 91, 291, 101, 21, 17, 289, 312, 282, 12, 133, 148, 318, 32, 18, 56, 307, 223, 317, 134, 164, 303, 285, 30, 86, 314, 119, 174, 178, 280, 162, 176, 84, 121, 166, 243, 136, 97, 278, 294, 298, 68, 290, 211, 233, 206, 207, 175, 90, 319, 193, 208, 173, 183, 115, 109, 197, 272, 273, 158, 35, 172, 28, 252, 296, 287, 204, 231, 257, 213, 126, 27, 255, 262, 142, 154, 235, 269, 202, 67, 253, 244, 160, 288, 209, 256, 301, 254, 218, 220, 99, 135, 292, 266, 81, 62, 171, 297, 265, 258, 195, 217, 203, 138, 293, 46, 320, 274, 310, 286, 281, 271, 189, 234, 305, 74, 196, 116, 141, 205, 279, 123, 95, 239, 155, 77, 131, 299, 120, 219, 98, 64, 88, 113, 316, 236, 31, 201, 47, 145, 309, 149, 284, 181, 275, 210, 188, 106, 75, 128, 40, 111, 308, 58, 57, 49, 78, 250, 92, 321, 215, 182, 267, 38, 228, 143, 227, 295, 107, 246, 122, 224, 190, 65, 94, 114,

,0,1,2,3,4,5,6,7,8,9,...,259,260,261,262,263,264,265,266,267,Hugo
0,37.670338,53.084359,67.978323,5.142525,28.779115,35.997939,-66.354566,42.128789,40.859722,-3.494659,...,2.533341,-4.194449,2.199545,2.105111,8.750158,6.434991,-2.003543,-1.455945,-1.582142,TCGA-BA-4074-01
1,18.747572,36.521362,52.967754,-0.986095,5.816400,20.203627,-34.478752,31.905618,29.976171,-1.679138,...,-5.219431,2.282592,0.923005,-6.399011,-3.564657,6.865756,1.837712,2.892176,2.122785,TCGA-BA-4075-01
2,-8.617162,21.559138,-3.561595,-9.908476,-27.873829,12.940778,-16.078410,8.703295,2.978251,5.997935,...,3.258308,-7.781988,-2.962046,-0.118432,-3.523021,0.418489,0.967021,1.660817,-1.348325,TCGA-BA-4076-01
3,10.794430,-33.773920,3.255420,-8.598195,-7.035513,6.982973,6.949812,17.515897,17.199286,4.231381,...,-1.598537,5.498055,-0.423117,2.446755,6.630633,1.385089,-1.578452,-1.254412,2.247790,TCGA-BA-4077-01
4,30.849528,0.859542,-31.948145,-9.522095,17.882788,-9.670178,4.653888,-0.572337,19.792068,-2.053513,...,-1.126224,-10.565322,-0.902627,1.107964,14.077208,-9.506386,-9.537101,2.171075,-1.131861,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,-8.064500,-6.504918,18.984889,-4.263380,4.774887,-0.753947,-5.278254,17.305591,-5.785138,6.093438,...,-0.381150,1.906592,-2.126816,-0.196648,0.755469,-0.317020,0.422109,1.780092,-0.647518,TCGA-UF-A7JT-01
515,-19.149224,6.577262,9.584200,-20.251779,5.542687,1.899250,-6.601919,13.511837,1.880846,-8.410102,...,-0.335162,4.882353,-0.655194,-0.742391,-5.988748,0.684287,-4.472186,0.286845,0.870929,TCGA-UF-A7JV-01
516,7.429759,-21.273752,-34.309695,-14.520433,-4.314327,8.153220,7.202251,22.823079,-9.384123,8.946533,...,1.898333,-19.010302,-3.795092,-6.020793,13.022279,-5.814478,-2.028380,-1.563740,1.976183,TCGA-UP-A6WW-01
517,-6.085431,20.345662,-20.180291,-0.179792,-17.785339,-14.722472,-4.287629,-2.094106,-7.230530,6.967738,...,0.058639,6.590292,-0.104680,-1.263299,0.309647,0.237559,1.189328,-3.503578,0.580321,TCGA-WA-A7GZ-01


In [22]:
lr = LinearRegression()
rfe = RFE(lr, n_features_to_select=291)
rfe.fit(Z3, y)
selected_features = [i+1 for i in range(len(rfe.support_)) if rfe.support_[i]]
print("Selected Features:", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z3[:, i]
        columns.append(column)
except Exception:
    pass
Zre3 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
nre3 = pd.DataFrame(Zre3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
nre3["Hugo"]=hugo
nre3

Selected Features: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 141, 142, 143, 144, 145, 149, 150, 152, 153, 158, 160, 161, 162, 164, 165, 166, 168, 170, 171, 172, 174, 175, 176, 179, 180, 182, 184, 188, 189, 190, 191, 192, 193, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241

,0,1,2,3,4,5,6,7,8,9,...,281,282,283,284,285,286,287,288,289,Hugo
0,-66.354566,37.670338,67.978323,28.779115,42.128789,5.142525,35.997939,40.859722,53.084359,24.707827,...,-0.743957,0.403235,0.606136,0.721832,-0.017289,-0.094052,-0.303523,-0.607368,0.004079,TCGA-BA-4074-01
1,-34.478752,18.747572,52.967754,5.816400,31.905618,-0.986095,20.203627,29.976171,36.521362,9.148691,...,-0.426687,0.730708,0.662740,0.770012,0.955220,0.470969,0.480922,0.577560,0.274844,TCGA-BA-4075-01
2,-16.078410,-8.617162,-3.561595,-27.873829,8.703295,-9.908476,12.940778,2.978251,21.559138,15.186483,...,-0.328034,0.393529,0.010860,-0.098502,0.129597,0.719325,0.288732,-0.224209,0.006392,TCGA-BA-4076-01
3,6.949812,10.794430,3.255420,-7.035513,17.515897,-8.598195,6.982973,17.199286,-33.773920,-7.168001,...,-1.205240,-1.195129,-1.515942,0.660014,0.651431,0.188518,0.381581,0.922257,0.367034,TCGA-BA-4077-01
4,4.653888,30.849528,-31.948145,17.882788,-0.572337,-9.522095,-9.670178,19.792068,0.859542,-6.248489,...,0.410865,-0.410290,-0.577895,-0.607617,-0.245760,-0.287860,1.031961,-1.287826,-1.785377,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,-5.278254,-8.064500,18.984889,4.774887,17.305591,-4.263380,-0.753947,-5.785138,-6.504918,4.687081,...,-1.463618,-1.023619,1.692489,-1.189679,-0.174098,0.207168,0.470725,-0.394500,0.428045,TCGA-UF-A7JT-01
515,-6.601919,-19.149224,9.584200,5.542687,13.511837,-20.251779,1.899250,1.880846,6.577262,-1.803487,...,0.676021,-0.765787,-0.335761,1.210208,-0.526118,0.765847,0.000097,0.943023,1.060078,TCGA-UF-A7JV-01
516,7.202251,7.429759,-34.309695,-4.314327,22.823079,-14.520433,8.153220,-9.384123,-21.273752,-11.644354,...,-0.892975,0.846924,-0.777825,-1.043758,-0.705052,-0.423263,-0.093982,-0.272106,-0.449100,TCGA-UP-A6WW-01
517,-4.287629,-6.085431,-20.180291,-17.785339,-2.094106,-0.179792,-14.722472,-7.230530,20.345662,-0.537323,...,1.356229,-0.605799,-0.951320,0.212750,1.258194,0.208172,0.720379,-0.263864,1.176720,TCGA-WA-A7GZ-01


In [23]:
en = ElasticNetCV(l1_ratio=0.5, cv=5)
en.fit(Z3, y)
coef = pd.Series(en.coef_, index=[i+1 for i in range(Z3.shape[1])])
selected_features = coef.abs().nlargest(291).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())
try:
    columns = []
    for i in selected_features:
        column = Z3[:, i]
        columns.append(column)
except Exception:
    pass
Ze3 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
ne3 = pd.DataFrame(Ze3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
ne3["Hugo"]=hugo
ne3


Number of selected features:  291
Selected features:  [7, 2, 9, 8, 6, 3, 5, 13, 14, 11, 4, 15, 12, 16, 10, 19, 18, 17, 1, 21, 20, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211

,0,1,2,3,4,5,6,7,8,9,...,282,283,284,285,286,287,288,289,290,Hugo
0,35.997939,37.670338,53.084359,40.859722,5.142525,67.978323,42.128789,-13.860106,-3.494659,20.802361,...,-1.455945,0.212366,-0.108858,0.270963,0.251042,0.085546,0.259345,1.116264,2.263076,TCGA-BA-4074-01
1,20.203627,18.747572,36.521362,29.976171,-0.986095,52.967754,31.905618,-5.683910,-1.679138,6.681710,...,2.892176,0.399405,-1.434089,3.486249,-3.878783,-1.533285,-0.775821,-1.688490,-3.631512,TCGA-BA-4075-01
2,12.940778,-8.617162,21.559138,2.978251,-9.908476,-3.561595,8.703295,-16.408898,5.997935,-0.461298,...,1.660817,0.568102,0.738011,-1.393226,1.204032,1.021035,1.118357,-2.051405,-3.580391,TCGA-BA-4076-01
3,6.982973,10.794430,-33.773920,17.199286,-8.598195,3.255420,17.515897,-4.377668,4.231381,-6.061248,...,-1.254412,-1.841860,1.425418,-1.759718,1.706544,0.186046,1.553687,-1.153301,-0.353455,TCGA-BA-4077-01
4,-9.670178,30.849528,0.859542,19.792068,-9.522095,-31.948145,-0.572337,-1.577633,-2.053513,8.252911,...,2.171075,2.025033,-1.564460,1.391341,0.003234,0.929930,0.987177,-2.956272,-1.767065,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,-0.753947,-8.064500,-6.504918,-5.785138,-4.263380,18.984889,17.305591,5.672817,6.093438,-4.561469,...,1.780092,2.056438,-2.573606,-4.812302,-4.178726,-0.418421,-3.139558,-4.212265,1.866548,TCGA-UF-A7JT-01
515,1.899250,-19.149224,6.577262,1.880846,-20.251779,9.584200,13.511837,-12.915732,-8.410102,-10.426937,...,0.286845,-3.531536,-6.561085,4.337769,0.664469,0.771246,2.467960,-2.248141,0.809502,TCGA-UF-A7JV-01
516,8.153220,7.429759,-21.273752,-9.384123,-14.520433,-34.309695,22.823079,-12.595259,8.946533,16.294026,...,-1.563740,-2.672683,-1.944337,0.204483,-0.304004,0.303763,0.566061,-0.372589,-1.308615,TCGA-UP-A6WW-01
517,-14.722472,-6.085431,20.345662,-7.230530,-0.179792,-20.180291,-2.094106,9.096657,6.967738,-18.466650,...,-3.503578,-0.103862,-0.520375,0.912064,-0.599100,-1.774910,0.266321,2.971117,1.780281,TCGA-WA-A7GZ-01


In [24]:
lasso = Lasso(alpha=0.1)  # alpha is the regularization strength
lasso.fit(Z3, y)
n_components = Z3.shape[1]
feature_names = [i+1 for i in range(n_components)]
coef = pd.Series(lasso.coef_, index=feature_names)
selected_features = coef.abs().nlargest(291).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z3[:, i]
        columns.append(column)
except Exception:
    pass
Zl3 = np.column_stack(columns)
nl3 = pd.DataFrame(Zl3)
hugo=[]
for i in range(1,520):
    hugo.append(drn['index'][i])
nl3["Hugo"]=hugo
nl3

Number of selected features:  291
Selected features:  Int64Index([  7,   2,   9,   8,   6,   3,   5,  14,  13,  11,
            ...
            282, 283, 284, 285, 286, 287, 288, 289, 290, 291],
           dtype='int64', length=291)


,0,1,2,3,4,5,6,7,8,9,...,282,283,284,285,286,287,288,289,290,Hugo
0,35.997939,37.670338,53.084359,40.859722,5.142525,67.978323,42.128789,-3.494659,-13.860106,20.802361,...,-1.455945,0.212366,-0.108858,0.270963,0.251042,0.085546,0.259345,1.116264,2.263076,TCGA-BA-4074-01
1,20.203627,18.747572,36.521362,29.976171,-0.986095,52.967754,31.905618,-1.679138,-5.683910,6.681710,...,2.892176,0.399405,-1.434089,3.486249,-3.878783,-1.533285,-0.775821,-1.688490,-3.631512,TCGA-BA-4075-01
2,12.940778,-8.617162,21.559138,2.978251,-9.908476,-3.561595,8.703295,5.997935,-16.408898,-0.461298,...,1.660817,0.568102,0.738011,-1.393226,1.204032,1.021035,1.118357,-2.051405,-3.580391,TCGA-BA-4076-01
3,6.982973,10.794430,-33.773920,17.199286,-8.598195,3.255420,17.515897,4.231381,-4.377668,-6.061248,...,-1.254412,-1.841860,1.425418,-1.759718,1.706544,0.186046,1.553687,-1.153301,-0.353455,TCGA-BA-4077-01
4,-9.670178,30.849528,0.859542,19.792068,-9.522095,-31.948145,-0.572337,-2.053513,-1.577633,8.252911,...,2.171075,2.025033,-1.564460,1.391341,0.003234,0.929930,0.987177,-2.956272,-1.767065,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
514,-0.753947,-8.064500,-6.504918,-5.785138,-4.263380,18.984889,17.305591,6.093438,5.672817,-4.561469,...,1.780092,2.056438,-2.573606,-4.812302,-4.178726,-0.418421,-3.139558,-4.212265,1.866548,TCGA-UF-A7JT-01
515,1.899250,-19.149224,6.577262,1.880846,-20.251779,9.584200,13.511837,-8.410102,-12.915732,-10.426937,...,0.286845,-3.531536,-6.561085,4.337769,0.664469,0.771246,2.467960,-2.248141,0.809502,TCGA-UF-A7JV-01
516,8.153220,7.429759,-21.273752,-9.384123,-14.520433,-34.309695,22.823079,8.946533,-12.595259,16.294026,...,-1.563740,-2.672683,-1.944337,0.204483,-0.304004,0.303763,0.566061,-0.372589,-1.308615,TCGA-UP-A6WW-01
517,-14.722472,-6.085431,20.345662,-7.230530,-0.179792,-20.180291,-2.094106,6.967738,9.096657,-18.466650,...,-3.503578,-0.103862,-0.520375,0.912064,-0.599100,-1.774910,0.266321,2.971117,1.780281,TCGA-WA-A7GZ-01


In [25]:
df = pd.read_csv("data_linear_CNA.csv")

#DROP ROWS AND COLUMNS
df =df.dropna()
df.drop("Entrez_Gene_Id", axis=1 ,inplace=True)

#SET INDEX
df.set_index('Hugo_Symbol', inplace=True)

#TRANSPOSE ROWS INTO COLUMNS
df5 = df.T
df5.to_csv("CNA.csv")
df5.head()
merged_df = pd.merge(cl, df5, left_index=True, right_index=True, how="inner")
df5=merged_df
c5 = pd.DataFrame(df5)
c5.to_csv("CNApreprocessed.csv")
y = c5['Overall Survival (Months)']
c5 = c5.drop('Overall Survival (Months)', axis=1)
y.drop(y.index[-1], inplace=True)
dcn=c5.iloc[:,:]
#print(d)
dcn = dcn.reset_index()
Cn=dcn.iloc[1:,1:]
Cn.head()
#PCA 
scaler = StandardScaler()
# Fit on training set only.
X4 = scaler.fit_transform(Cn[:])
#fit pca on data
pca = PLSRegression(n_components=161)
pca.fit(X4,y)

#transform pca
Z4 =pca.transform(X4)
Z4

n4 = pd.DataFrame(Z4)
hugo=[]
for i in range(1,522):
    hugo.append(dcn['index'][i])
n4["Hugo"]=hugo
n4

,0,1,2,3,4,5,6,7,8,9,...,152,153,154,155,156,157,158,159,160,Hugo
0,-19.140882,-0.208776,-21.916783,-26.015449,25.147111,9.868919,23.815222,2.019978,3.518508,3.964065,...,1.706471,2.979605,-3.006542,-3.741260,-2.674967,1.733277,1.172647,-7.806259,-6.382815,TCGA-BA-4074-01
1,23.240997,-30.770335,4.360017,-18.313325,-22.047382,6.709602,31.244491,-38.397833,5.036911,-52.818855,...,3.143746,-1.253696,1.554784,2.686111,-0.652956,3.347652,-0.885432,4.053074,1.871508,TCGA-BA-4075-01
2,-46.218994,-52.435015,-2.302688,15.563072,55.745632,14.411289,17.035305,-18.254456,16.215675,-4.879511,...,-1.018329,2.680677,3.369367,-0.949042,-2.281421,2.119660,-0.441597,1.560447,-2.417399,TCGA-BA-4076-01
3,-26.204122,-26.943317,14.719854,-5.749955,-17.213696,-11.741620,-10.177180,-3.751015,3.882267,26.792640,...,-5.295062,9.061900,2.261704,6.857163,-3.884426,-2.499627,1.344145,-2.589695,6.937258,TCGA-BA-4077-01
4,4.367695,92.380108,31.653031,-80.942807,11.313076,-10.245502,-6.496756,15.784056,-33.906600,31.197665,...,1.525710,-3.492718,0.686697,-4.449343,2.144852,1.302575,2.863171,-3.735232,4.305530,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
516,-11.680594,3.478905,1.669287,14.576911,-17.819679,-0.577177,1.470767,2.979586,-5.439101,-4.286271,...,-1.015565,-0.460972,-0.115845,0.760703,2.083706,-1.652540,-2.950899,0.765679,0.380447,TCGA-UF-A7JT-01
517,-1.400661,8.137265,-10.608284,9.677937,-18.441801,7.026115,1.691049,0.325782,2.631081,-11.961765,...,1.917371,-0.988611,0.812447,-1.122741,-2.751158,1.303093,-1.590156,1.307823,1.771739,TCGA-UF-A7JV-01
518,-44.064176,47.409459,-13.926291,-37.633848,-16.935389,-3.447494,-6.089212,18.176349,-21.818937,15.272742,...,0.578390,1.018946,-0.462786,1.318209,-2.553351,2.721572,1.962485,1.077917,-3.208558,TCGA-UP-A6WW-01
519,-7.232954,9.301045,-6.294279,-54.155738,-13.379795,3.043947,10.976672,-4.564463,36.123978,-14.957262,...,-2.877497,-0.220319,1.897182,0.418614,3.886636,3.991188,3.194094,5.225586,1.323172,TCGA-WA-A7GZ-01


In [26]:
lasso = Lasso(alpha=0.1)  # alpha is the regularization strength
lasso.fit(Z4, y)
n_components = Z4.shape[1]
feature_names = [i+1 for i in range(n_components)]
coef = pd.Series(lasso.coef_, index=feature_names)
selected_features = coef.abs().nlargest(144).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z4[:, i]
        columns.append(column)
except Exception:
    pass
Zl4 = np.column_stack(columns)
nl4 = pd.DataFrame(Zl4)
hugo=[]
for i in range(1,522):
    hugo.append(dcn['index'][i])
nl4["Hugo"]=hugo
nl4

Number of selected features:  144
Selected features:  Int64Index([  7,   3,   6,  12,   1,   9,   2,   4,  14,  11,
            ...
            133, 132, 130, 140, 129, 139, 142, 141, 144, 146],
           dtype='int64', length=144)


,0,1,2,3,4,5,6,7,8,9,...,135,136,137,138,139,140,141,142,143,Hugo
0,2.019978,-26.015449,23.815222,-4.860367,-0.208776,3.964065,-21.916783,25.147111,-34.894566,6.582809,...,-0.877770,3.526999,-1.823161,2.988491,-1.521368,0.074093,-1.904947,-1.346519,-0.233027,TCGA-BA-4074-01
1,-38.397833,-18.313325,31.244491,23.256085,-30.770335,-52.818855,4.360017,-22.047382,-7.023387,26.516207,...,-1.093691,2.891138,-1.283698,0.611640,-3.470408,-1.391411,3.125826,6.563952,0.665564,TCGA-BA-4075-01
2,-18.254456,15.563072,17.035305,5.791840,-52.435015,-4.879511,-2.302688,55.745632,-13.450334,-4.428115,...,5.793096,3.147889,4.351346,-0.953358,-0.210929,-4.801245,0.769308,-6.572000,4.226729,TCGA-BA-4076-01
3,-3.751015,-5.749955,-10.177180,28.124332,-26.943317,26.792640,14.719854,-17.213696,5.438071,-0.903876,...,2.850492,5.477267,-0.180639,-7.026858,2.701209,2.949925,2.128164,-3.893939,-0.030460,TCGA-BA-4077-01
4,15.784056,-80.942807,-6.496756,-13.187321,92.380108,31.197665,31.653031,11.313076,-20.457747,-19.572851,...,-4.116640,-0.433586,0.131719,2.006711,5.046347,-0.362599,-3.925167,-0.972678,-7.910659,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
516,2.979586,14.576911,1.470767,2.353460,3.478905,-4.286271,1.669287,-17.819679,4.720390,-3.330367,...,-0.683888,1.112740,0.403688,1.116015,1.441669,-1.619328,-0.291156,-1.726218,0.877022,TCGA-UF-A7JT-01
517,0.325782,9.677937,1.691049,-5.415278,8.137265,-11.961765,-10.608284,-18.441801,3.843462,-2.888955,...,-0.088495,2.099687,1.990157,2.015068,-2.910929,1.058004,0.809332,1.745085,-1.416989,TCGA-UF-A7JV-01
518,18.176349,-37.633848,-6.089212,-36.185897,47.409459,15.272742,-13.926291,-16.935389,-1.392192,5.271113,...,-2.091716,2.854279,-4.843035,3.103077,7.531140,-1.356568,-1.794492,-3.485966,8.997825,TCGA-UP-A6WW-01
519,-4.564463,-54.155738,10.976672,-18.345900,9.301045,-14.957262,-6.294279,-13.379795,12.275905,-0.084376,...,3.864308,-0.980130,-4.095105,-0.900182,-2.878889,7.090598,2.905772,-0.752284,0.622485,TCGA-WA-A7GZ-01


In [27]:
en = ElasticNetCV(l1_ratio=0.5, cv=5)
en.fit(Z4, y)
coef = pd.Series(en.coef_, index=[i+1 for i in range(Z4.shape[1])])
selected_features = coef.abs().nlargest(144).index
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())
try:
    columns = []
    for i in selected_features:
        column = Z4[:, i]
        columns.append(column)
except Exception:
    pass
Ze4 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
ne4 = pd.DataFrame(Ze4)
hugo=[]
try:
    for i in range(1,522):
        hugo.append(dcn['index'][i])
    ne4["Hugo"]=hugo
except Exception:
    pass


Number of selected features:  144
Selected features:  [7, 3, 6, 12, 1, 9, 2, 4, 14, 11, 16, 15, 18, 10, 22, 8, 17, 41, 20, 37, 30, 5, 19, 13, 26, 21, 25, 28, 34, 59, 35, 49, 24, 23, 62, 43, 31, 27, 68, 47, 36, 33, 64, 82, 69, 40, 61, 50, 65, 79, 57, 45, 39, 67, 52, 70, 42, 56, 53, 87, 78, 58, 72, 60, 73, 83, 38, 86, 76, 93, 32, 71, 81, 88, 46, 48, 66, 29, 63, 84, 91, 85, 51, 77, 74, 90, 44, 55, 54, 80, 89, 98, 97, 75, 99, 102, 100, 94, 96, 95, 101, 103, 92, 104, 107, 110, 105, 109, 111, 106, 112, 116, 123, 114, 120, 113, 117, 124, 115, 121, 126, 137, 108, 125, 122, 127, 119, 138, 118, 128, 131, 134, 135, 136, 133, 132, 130, 140, 129, 139, 142, 141, 144, 146]


In [28]:
lr = LinearRegression()
rfe = RFE(lr, n_features_to_select=144)
rfe.fit(Z4, y)
selected_features = [i+1 for i in range(len(rfe.support_)) if rfe.support_[i]]
print("Selected Features:", selected_features)
try:
    columns = []
    for i in selected_features:
        column = Z4[:, i]
        columns.append(column)
except Exception:
    pass
Zre4 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
nre4 = pd.DataFrame(Zre4)
hugo=[]
try:
    for i in range(1,522):
        hugo.append(dcn['index'][i])
    nre4["Hugo"]=hugo
except Exception:
    pass

Selected Features: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 144, 146]


In [29]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(Z4, y)
importances = rf.feature_importances_
feature_names = [i+1 for i in range(Z4.shape[1])]
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
importance_df.sort_values("Importance", ascending=False, inplace=True)
selected_features = importance_df["Feature"].head(144)
print("Number of selected features: ", len(selected_features))
print("Selected features: ", selected_features.tolist())

try:
    columns = []
    for i in selected_features:
        column = Z4[:, i]
        columns.append(column)
except Exception:
    pass
Zrf4 = np.column_stack(columns)
#STORING REDUCED DATA TO CSV
nrf4 = pd.DataFrame(Zrf4)
hugo=[]
for i in range(1,522):
    hugo.append(dcn['index'][i])
nrf4["Hugo"]=hugo
nrf4

Number of selected features:  144
Selected features:  [4, 2, 6, 9, 3, 7, 84, 97, 11, 91, 12, 20, 1, 82, 85, 86, 25, 23, 21, 5, 65, 14, 60, 18, 53, 17, 8, 24, 16, 22, 29, 28, 13, 68, 26, 96, 55, 30, 59, 127, 92, 118, 43, 37, 110, 40, 39, 153, 111, 10, 19, 42, 66, 72, 38, 57, 15, 27, 46, 102, 116, 44, 63, 34, 35, 117, 32, 47, 147, 87, 103, 138, 144, 70, 160, 31, 146, 152, 36, 93, 113, 79, 77, 101, 73, 67, 121, 33, 54, 115, 51, 52, 41, 89, 99, 71, 149, 98, 49, 95, 131, 120, 56, 125, 58, 106, 108, 45, 129, 136, 134, 94, 114, 155, 80, 142, 64, 140, 141, 112, 137, 139, 132, 143, 90, 100, 154, 157, 61, 50, 130, 150, 124, 156, 161, 128, 76, 74, 148, 107, 109, 75, 151, 69]


,0,1,2,3,4,5,6,7,8,9,...,125,126,127,128,129,130,131,132,133,Hugo
0,25.147111,-21.916783,23.815222,3.964065,-26.015449,2.019978,0.028958,3.174478,6.582809,-2.742268,...,-0.684816,-3.006542,1.733277,-1.370228,-8.561804,3.526999,-1.200843,0.229817,-2.674967,TCGA-BA-4074-01
1,-22.047382,4.360017,31.244491,-52.818855,-18.313325,-38.397833,2.211000,-6.428666,26.516207,0.826670,...,2.311905,1.554784,3.347652,-2.500914,0.138043,2.891138,1.514046,-4.165093,-0.652956,TCGA-BA-4075-01
2,55.745632,-2.302688,17.035305,-4.879511,15.563072,-18.254456,0.212223,-16.334708,-4.428115,34.117654,...,-1.958440,3.369367,2.119660,9.064284,-44.773103,3.147889,1.379377,0.906341,-2.281421,TCGA-BA-4076-01
3,-17.213696,14.719854,-10.177180,26.792640,-5.749955,-3.751015,2.551481,-3.142733,-0.903876,2.083620,...,0.270340,2.261704,-2.499627,9.693577,-7.960215,5.477267,-11.324632,-2.826059,-3.884426,TCGA-BA-4077-01
4,11.313076,31.653031,-6.496756,31.197665,-80.942807,15.784056,-5.461055,-0.765544,-19.572851,6.428670,...,-0.583740,0.686697,1.302575,1.968794,-34.388313,-0.433586,-0.460266,-0.708086,2.144852,TCGA-BA-4078-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
516,-17.819679,1.669287,1.470767,-4.286271,14.576911,2.979586,3.391046,0.814938,-3.330367,0.709207,...,4.467577,-0.115845,-1.652540,0.170569,0.542697,1.112740,0.017854,-4.758603,2.083706,TCGA-UF-A7JT-01
517,-18.441801,-10.608284,1.691049,-11.961765,9.677937,0.325782,-2.026537,0.854973,-2.888955,1.057543,...,1.138239,0.812447,1.303093,1.904996,2.998112,2.099687,-0.174601,-3.559193,-2.751158,TCGA-UF-A7JV-01
518,-16.935389,-13.926291,-6.089212,15.272742,-37.633848,18.176349,0.611158,-3.499243,5.271113,1.150739,...,2.182244,-0.462786,2.721572,-1.872109,6.507913,2.854279,-0.151034,-5.480505,-2.553351,TCGA-UP-A6WW-01
519,-13.379795,-6.294279,10.976672,-14.957262,-54.155738,-4.564463,-8.229572,9.566326,-0.084376,1.735144,...,-0.995402,1.897182,3.991188,-9.621056,-1.188931,-0.980130,1.947323,-2.627227,3.886636,TCGA-WA-A7GZ-01


Merging dataset

In [31]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(n1['Hugo'][i]==n3['Hugo'][j]):
            z.append(n1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==n4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,n1,on='Hugo')
mdf=pd.merge(mdf,n3,on='Hugo')
mdf=pd.merge(mdf,n4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_val=df_val
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
Y_val = get_target(df_val)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)
E_val = get_target(df_val)
# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)


epochs = 30
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  validation_data=(X_val, Y_val),
                  epochs=epochs,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DEEPSURV")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)




(512, 26)
(512, 872)
Epoch 1/30
1/1 [==============================] - 26s 26s/step - loss: 569087.3125 - val_loss: 101523.7578
Epoch 2/30
1/1 [==============================] - 11s 11s/step - loss: 567619.2500 - val_loss: 101523.7188
Epoch 3/30
1/1 [==============================] - 9s 9s/step - loss: 567646.1250 - val_loss: 101523.6797
Epoch 4/30
1/1 [==============================] - 8s 8s/step - loss: 567230.8125 - val_loss: 101523.6406
Epoch 5/30
1/1 [==============================] - 8s 8s/step - loss: 567272.0000 - val_loss: 101523.6094
Epoch 6/30
1/1 [==============================] - 8s 8s/step - loss: 568885.5625 - val_loss: 101523.5703
Epoch 7/30
1/1 [==============================] - 8s 8s/step - loss: 567266.8125 - val_loss: 101523.5312
Epoch 8/30
1/1 [==============================] - 8s 8s/step - loss: 567003.1250 - val_loss: 101523.4922
Epoch 9/30
1/1 [==============================] - 8s 8s/step - loss: 578436.9375 - val_loss: 101523.4531
Epoch 10/30
1/1 [=============

In [35]:
epochs = 100
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  validation_data=(X_val, Y_val),
                  epochs=epochs,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DEEPSURV")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)

Epoch 1/100
1/1 [==============================] - 15s 15s/step - loss: 443715.3438 - val_loss: 101453.8438
Epoch 2/100
1/1 [==============================] - 10s 10s/step - loss: 442350.0625 - val_loss: 101449.5312
Epoch 3/100
1/1 [==============================] - 8s 8s/step - loss: 448731.2812 - val_loss: 101446.9062
Epoch 4/100
1/1 [==============================] - 8s 8s/step - loss: 444011.0625 - val_loss: 101446.0234
Epoch 5/100
1/1 [==============================] - 8s 8s/step - loss: 451414.0938 - val_loss: 101439.3438
Epoch 6/100
1/1 [==============================] - 8s 8s/step - loss: 450401.5312 - val_loss: 101429.3828
Epoch 7/100
1/1 [==============================] - 8s 8s/step - loss: 440788.7188 - val_loss: 101412.5859
Epoch 8/100
1/1 [==============================] - 8s 8s/step - loss: 444461.7500 - val_loss: 101398.7188
Epoch 9/100
1/1 [==============================] - 8s 8s/step - loss: 444573.4062 - val_loss: 101390.0078
Epoch 10/100
1/1 [========================

1/1 [==============================] - 8s 8s/step - loss: 446491.8438 - val_loss: 101277.8828
Epoch 78/100
1/1 [==============================] - 7s 7s/step - loss: 443063.3438 - val_loss: 101294.4219
Epoch 79/100
1/1 [==============================] - 8s 8s/step - loss: 442700.9688 - val_loss: 101309.7656
Epoch 80/100
1/1 [==============================] - 7s 7s/step - loss: 444057.3750 - val_loss: 101312.5078
Epoch 81/100
1/1 [==============================] - 8s 8s/step - loss: 448174.1250 - val_loss: 101298.8750
Epoch 82/100
1/1 [==============================] - 8s 8s/step - loss: 448832.0625 - val_loss: 101303.7031
Epoch 83/100
1/1 [==============================] - 8s 8s/step - loss: 447648.8438 - val_loss: 101309.6172
Epoch 84/100
1/1 [==============================] - 8s 8s/step - loss: 448230.1875 - val_loss: 101295.1875
Epoch 85/100
1/1 [==============================] - 8s 8s/step - loss: 452079.3125 - val_loss: 101277.4688
Epoch 86/100
1/1 [==============================] 

In [ ]:
import deepsurvk

dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)
callbacks = deepsurvk.common_callbacks()
print(callbacks)

epochs = 20
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  epochs=epochs, 
                  callbacks=callbacks,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DEEPSURV")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)


In [ ]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)


In [ ]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)

In [36]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(nl1['Hugo'][i]==nl3['Hugo'][j]):
            z.append(nl1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==nl4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,nl1,on='Hugo')
mdf=pd.merge(mdf,nl3,on='Hugo')
mdf=pd.merge(mdf,nl4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)

X_val=df_val
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
Y_val = get_target(df_val)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)
E_val = get_target(df_val)
# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)


epochs = 30
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  validation_data=(X_val, Y_val),
                  epochs=epochs,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DeepSurv)")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)



(512, 26)
(512, 784)
Epoch 1/30
1/1 [==============================] - 23s 23s/step - loss: 621146.0000 - val_loss: 110987.0547
Epoch 2/30
1/1 [==============================] - 6s 6s/step - loss: 617465.1250 - val_loss: 110987.0078
Epoch 3/30
1/1 [==============================] - 7s 7s/step - loss: 663922.9375 - val_loss: 110986.9609
Epoch 4/30
1/1 [==============================] - 7s 7s/step - loss: 617654.1250 - val_loss: 110986.9219
Epoch 5/30
1/1 [==============================] - 7s 7s/step - loss: 620000.5625 - val_loss: 110986.8828
Epoch 6/30
1/1 [==============================] - 8s 8s/step - loss: 618323.8750 - val_loss: 110986.8438
Epoch 7/30
1/1 [==============================] - 7s 7s/step - loss: 638893.3125 - val_loss: 110986.7969
Epoch 8/30
1/1 [==============================] - 7s 7s/step - loss: 619777.3125 - val_loss: 110986.7578
Epoch 9/30
1/1 [==============================] - 7s 7s/step - loss: 620146.8125 - val_loss: 110986.7188
Epoch 10/30
1/1 [===============

In [37]:
epochs = 100
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  validation_data=(X_val, Y_val),
                  epochs=epochs,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DEEPSURV")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)

Epoch 1/100
1/1 [==============================] - 7s 7s/step - loss: 620145.8750 - val_loss: 111193.7578
Epoch 2/100
1/1 [==============================] - 6s 6s/step - loss: 620145.8125 - val_loss: 111177.5234
Epoch 3/100
1/1 [==============================] - 6s 6s/step - loss: 620145.8125 - val_loss: 111158.6094
Epoch 4/100
1/1 [==============================] - 7s 7s/step - loss: 619632.5000 - val_loss: 111147.7031
Epoch 5/100
1/1 [==============================] - 7s 7s/step - loss: 620145.6875 - val_loss: 111134.7109
Epoch 6/100
1/1 [==============================] - 6s 6s/step - loss: 620145.6250 - val_loss: 111118.1797
Epoch 7/100
1/1 [==============================] - 6s 6s/step - loss: 620145.6250 - val_loss: 111101.2969
Epoch 8/100
1/1 [==============================] - 7s 7s/step - loss: 620145.5625 - val_loss: 111090.5547
Epoch 9/100
1/1 [==============================] - 8s 8s/step - loss: 620145.5000 - val_loss: 111078.6875
Epoch 10/100
1/1 [============================

1/1 [==============================] - 7s 7s/step - loss: 612935.9375 - val_loss: 110973.1875
Epoch 78/100
1/1 [==============================] - 7s 7s/step - loss: 612389.3125 - val_loss: 110974.5156
Epoch 79/100
1/1 [==============================] - 6s 6s/step - loss: 609695.7500 - val_loss: 110975.1250
Epoch 80/100
1/1 [==============================] - 7s 7s/step - loss: 605991.6875 - val_loss: 110975.7734
Epoch 81/100
1/1 [==============================] - 6s 6s/step - loss: 603221.4375 - val_loss: 110975.7578
Epoch 82/100
1/1 [==============================] - 7s 7s/step - loss: 597281.6250 - val_loss: 110975.2422
Epoch 83/100
1/1 [==============================] - 6s 6s/step - loss: 596324.8750 - val_loss: 110978.7578
Epoch 84/100
1/1 [==============================] - 7s 7s/step - loss: 590069.5000 - val_loss: 110981.5078
Epoch 85/100
1/1 [==============================] - 7s 7s/step - loss: 585197.3125 - val_loss: 110987.1328
Epoch 86/100
1/1 [==============================] 

In [ ]:
result = np.zeros((5, 4, 5))

result[0][0][0]=c_index_test
result[0][0][1]=mse
result[0][0][2]=rmse
result[0][0][3]=mae
result[0][0][4]=mdae

In [ ]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)
result[1][0][0]=c_index
result[1][0][1]=mse
result[1][0][2]=r2
result[1][0][3]=mae
result[1][0][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[2][0][0]=c_index
result[2][0][1]=mse
result[2][0][2]=r2
result[2][0][3]=mae
result[2][0][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[3][0][0]=c_index
result[3][0][1]=mse
result[3][0][2]=r2
result[3][0][3]=mae
result[3][0][4]=mdae

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[4][0][0]=c_index
result[4][0][1]=mse
result[4][0][2]=r2
result[4][0][3]=mae
result[4][0][4]=mdae

In [ ]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(ne1['Hugo'][i]==ne3['Hugo'][j]):
            z.append(ne1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==ne4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,ne1,on='Hugo')
mdf=pd.merge(mdf,ne3,on='Hugo')
mdf=pd.merge(mdf,ne4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_val=df_val
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
Y_val = get_target(df_val)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)
E_val = get_target(df_val)
# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)


epochs = 30
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  validation_data=(X_val, Y_val),
                  epochs=epochs,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DeepSurv")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)
result[0][1][0]=c_index_test
result[0][1][1]=mse
result[0][1][2]=rmse
result[0][1][3]=mae
result[0][1][4]=mdae



In [ ]:
epochs = 100
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  validation_data=(X_val, Y_val),
                  epochs=epochs,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DEEPSURV")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)

In [ ]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)
result[1][1][0]=c_index
result[1][1][1]=mse
result[1][1][2]=r2
result[1][1][3]=mae
result[1][1][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[2][1][0]=c_index
result[2][1][1]=mse
result[2][1][2]=r2
result[2][1][3]=mae
result[2][1][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[3][1][0]=c_index
result[3][1][1]=mse
result[3][1][2]=r2
result[3][1][3]=mae
result[3][1][4]=mdae

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[4][1][0]=c_index
result[4][1][1]=mse
result[4][1][2]=r2
result[4][1][3]=mae
result[4][1][4]=mdae

In [ ]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(nre1['Hugo'][i]==nre3['Hugo'][j]):
            z.append(nre1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==nre4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,nre1,on='Hugo')
mdf=pd.merge(mdf,nre3,on='Hugo')
mdf=pd.merge(mdf,ne4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_val=df_val
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
Y_val = get_target(df_val)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)
E_val = get_target(df_val)
# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)


epochs = 30
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  validation_data=(X_val, Y_val),
                  epochs=epochs,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DeepSurv")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)
result[0][2][0]=c_index_test
result[0][2][1]=mse
result[0][2][2]=rmse
result[0][2][3]=mae
result[0][2][4]=mdae



In [ ]:
epochs = 100
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  validation_data=(X_val, Y_val),
                  epochs=epochs,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DEEPSURV")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)

In [ ]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)
result[1][2][0]=c_index
result[1][2][1]=mse
result[1][2][2]=r2
result[1][2][3]=mae
result[1][2][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[2][2][0]=c_index
result[2][2][1]=mse
result[2][2][2]=r2
result[2][2][3]=mae
result[2][2][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[3][2][0]=c_index
result[3][2][1]=mse
result[3][2][2]=r2
result[3][2][3]=mae
result[3][2][4]=mdae

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[4][2][0]=c_index
result[4][2][1]=mse
result[4][2][2]=r2
result[4][2][3]=mae
result[4][2][4]=mdae

In [ ]:
z=[]
x=[]

for i in range(0,527):
    for j in range(0,519):
        if(nrf1['Hugo'][i]==nrf3['Hugo'][j]):
            z.append(nrf1['Hugo'][i])
for i in range(0,len(z)):
    for j in range(0,520):
        if(z[i]==nrf4['Hugo'][j]):
            x.append(z[i])

dfmerge = pd.DataFrame()
dfmerge['Hugo']=x

mdf=pd.merge(dfmerge,nrf1,on='Hugo')
mdf=pd.merge(mdf,nrf3,on='Hugo')
mdf=pd.merge(mdf,nrf4,on='Hugo')

mdf = pd.DataFrame(mdf)
cl=pd.read_csv('CLINICALpreprocessed.CSV')
intersect= set.intersection(set(cl.index),set(mdf.index))
set(mdf.index)-intersect
for i in (set(mdf.index)-intersect):
    mdf.drop(i,inplace=True)
for i in (set(cl.index)-intersect):
    cl.drop(i,inplace=True)

cl.drop("Patient Identifier", axis=1,inplace=True)


print(cl.shape)
print(mdf.shape)
mdf=pd.concat([mdf,cl],axis=1)
mdf.drop("Hugo", axis=1,inplace=True)
mdf.columns = mdf.columns.astype(str)
scaler=per.MinMaxScaler(feature_range=(0,1))
rescaleData=scaler.fit_transform(mdf)
mdf=pd.DataFrame(rescaleData,index=mdf.index,columns=mdf.columns)
mdf.head()
df_train = mdf
df_test = df_train.sample(frac=0.3)
X_test=df_test
df_train = df_train.drop(df_test.index)
df_val = df_train.sample(frac=0.2)
X_val=df_val
X_train=df_train = df_train.drop(df_val.index)
df_train
get_target = lambda df: (df['Overall Survival (Months)'].values)
Y_train = get_target(df_train)

Y_test = get_target(df_test)
Y_val = get_target(df_val)
get_target = lambda df: (df['Overall Survival Status'].values)
E_train = get_target(df_train)
E_test = get_target(df_test)
E_val = get_target(df_val)
# Exclude non-numeric columns from X_train and X_test
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
n_features = X_train.shape[1]
n_patients_train = X_train.shape[0]


# Drop columns containing string values from X_train and X_test
columns_to_drop = X_train.columns[X_train.applymap(type).eq(str).any()]
if 'Hugo' in columns_to_drop:
    columns_to_drop = columns_to_drop.drop('Hugo')

X_train_numeric = X_train_numeric.drop(columns=columns_to_drop, errors='ignore')
X_test_numeric = X_test_numeric.drop(columns=columns_to_drop, errors='ignore')

# Sorting
sort_idx = np.argsort(Y_train)[::-1]

# Use array indexing to select rows based on sort_idx
X_train_sorted = X_train_numeric.iloc[sort_idx, :]
Y_train_sorted = Y_train[sort_idx]


dsk = deepsurvk.DeepSurvK(n_features=n_features, E=E_train,)


epochs = 30
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  validation_data=(X_val, Y_val),
                  epochs=epochs,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DeepSurv")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)
result[0][3][0]=c_index_test
result[0][3][1]=mse
result[0][3][2]=rmse
result[0][3][3]=mae
result[0][3][4]=mdae



In [ ]:
epochs = 100
history = dsk.fit(X_train, Y_train, 
                  batch_size=n_patients_train,
                  validation_data=(X_val, Y_val),
                  epochs=epochs,
                  shuffle=False)
Y_pred_train = np.exp(-dsk.predict(X_train))
c_index_train = deepsurvk.concordance_index(Y_train, Y_pred_train, E_train)
print("DEEPSURV")
print(f"c-index of training dataset = {c_index_train}")

Y_pred_test = np.exp(-dsk.predict(X_test))
c_index_test = deepsurvk.concordance_index(Y_test, Y_pred_test, E_test)
print(f"c-index of testing dataset = {c_index_test}")
Y_pred_trainn = (-dsk.predict(X_train))
Y_pred_testt = (-dsk.predict(X_test))

from sklearn.metrics import mean_squared_error
mse=mean_squared_error(Y_test,Y_pred_testt)

import math
rmse=math.sqrt(mse)

from sklearn.metrics import mean_absolute_error 
mae=mean_absolute_error (Y_test,Y_pred_testt)

from sklearn.metrics import median_absolute_error
mdae=median_absolute_error (Y_test,Y_pred_testt)

print("mse=",mse)
print("rmse=",rmse)
print("mae=",mae)
print("mdae=",mdae)

In [ ]:
param_grid = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf", "poly"],
    "degree": [2, 3, 4],
    "epsilon": [0.01, 0.1, 1],
    "gamma": ["scale", "auto"]
}

svr = SVR()
grid_search = GridSearchCV(svr, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
print("SVR")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
print("R-squared: ", r2)
result[1][3][0]=c_index
result[1][3][1]=mse
result[1][3][2]=r2
result[1][3][3]=mae
result[1][3][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 1],
    "max_depth": [1, 2, 3, 4]
}
xgb = XGBRegressor()
grid_search = GridSearchCV(xgb, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("XGBOOST")
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[2][3][0]=c_index
result[2][3][1]=mse
result[2][3][2]=r2
result[2][3][3]=mae
result[2][3][4]=mdae

In [ ]:
param_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.01, 0.1],
    "base_estimator__max_depth": [1, 2]
}
base_estimator = DecisionTreeRegressor()
adaboost = AdaBoostRegressor(base_estimator=base_estimator)
grid_search = GridSearchCV(adaboost, param_grid, cv=5)
grid_search.fit(X_train, Y_train)
y_pred = grid_search.predict(X_test)
mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[3][3][0]=c_index
result[3][3][1]=mse
result[3][3][2]=r2
result[3][3][3]=mae
result[3][3][4]=mdae

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, Y_train)
y_pred = rf.predict(X_test)

mse = mean_squared_error(Y_test, y_pred)
r2 = r2_score(Y_test, y_pred)
c_index = concordance_index(Y_test, y_pred)
mae = mean_absolute_error(Y_test, y_pred)
medae = median_absolute_error(Y_test, y_pred)
print("C-index: ", c_index)
print("Mean squared error: ", mse)
print("R-squared: ", r2)
print("c_index:",c_index)
print("Mean absolute error: ", mae)
print("Median absolute error: ", medae)
result[4][3][0]=c_index
result[4][3][1]=mse
result[4][3][2]=r2
result[4][3][3]=mae
result[4][3][4]=mdae

In [ ]:
!pip install pandas openpyxl
data_0_0 = result[0, 0:4, 0]
data_1_0 = result[3, 0:4, 0]
data_2_0 = result[2, 0:4, 0]
data_3_0 = result[1, 0:4, 0]
data_4_0 = result[4, 0:4, 0]
combined_data_0 = np.concatenate((data_0_0, data_1_0, data_2_0, data_3_0, data_4_0))
data_0_1 = result[0, 0:4, 1]
data_1_1 = result[3, 0:4, 1]
data_2_1 = result[2, 0:4, 1]
data_3_1 = result[1, 0:4, 1]
data_4_1 = result[4, 0:4, 1]
combined_data_1 = np.concatenate((data_0_1, data_1_1, data_2_1, data_3_1, data_4_1))
data_0_2 = result[0, 0:4, 2]
data_1_2 = result[3, 0:4, 2]
data_2_2 = result[2, 0:4, 2]
data_3_2 = result[1, 0:4, 2]
data_4_2 = result[4, 0:4, 2]
combined_data_2 = np.concatenate((data_0_2, data_1_2, data_2_2, data_3_2, data_4_2))
data_0_3 = result[0, 0:4, 3]
data_1_3 = result[3, 0:4, 3]
data_2_3 = result[2, 0:4, 3]
data_3_3 = result[1, 0:4, 3]
data_4_3 = result[4, 0:4, 3]
combined_data_3 = np.concatenate((data_0_3, data_1_3, data_2_3, data_3_3, data_4_3))
data_0_4 = result[0, 0:4, 4]
data_1_4 = result[3, 0:4, 4]
data_2_4 = result[2, 0:4, 4]
data_3_4 = result[1, 0:4, 4]
data_4_4 = result[4, 0:4, 4]
combined_data_4 = np.concatenate((data_0_4, data_1_4, data_2_4, data_3_4, data_4_4))
# Create a DataFrame
df = pd.DataFrame({
    'Combined_Data_0': combined_data_0,
    'Combined_Data_1': combined_data_1,
    'Combined_Data_2': combined_data_2,
    'Combined_Data_3': combined_data_3,
    'Combined_Data_4': combined_data_4
})

# Save the DataFrame to an Excel file
df.to_excel('combined_data.xlsx', index=False)

print("Data has been saved to combined_data.xlsx")